This notebook is a basic tutorial that demonstrates how to configure a simulation using Concordia.

<a href="https://colab.research.google.com/github/google-deepmind/concordia/blob/main/examples/tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# @title Colab-specific setup (use a CodeSpace to avoid the need for this).
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/google-deepmind/concordia/main/examples/requirements.in' 'git+https://github.com/google-deepmind/concordia.git#egg=gdm-concordia'
  %pip list

In [3]:
# @title Imports

from concordia.contrib import language_models as language_model_utils
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.prefabs.simulation import generic as simulation
from concordia.typing import prefab as prefab_lib
from concordia.utils import helper_functions
from IPython import display
import numpy as np
import sentence_transformers

/Users/taijusanagi/Documents/Developments/taijusanagi/concordia/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1118 17:50:16.589000 65288 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [4]:
# @title Language Model Selection: provide key or select DISABLE_LANGUAGE_MODEL

# By default this colab uses models via an external API so you must provide an
# API key. TogetherAI offers open weights models from all sources.

API_KEY = ''  # @param {type: 'string'}
# See concordia/language_model/utils.py
API_TYPE = 'openai_codex'  # e.g. 'together_ai', 'openai' or 'gemini'.
# Check all supported models in _REGISTRY concordia/contrib/language_models/__init__.py
MODEL_NAME = (  # for API_TYPE = 'together_ai', we recommend MODEL_NAME = 'google/gemma-3-27b-it'
    'gpt-6-luna'
)
# To debug without spending money on API calls, set DISABLE_LANGUAGE_MODEL=True
DISABLE_LANGUAGE_MODEL = False

In [5]:
# @title Use the selected language model

# Note that it is also possible to use local models or other API models,
# simply replace this cell with the correct initialization for the model
# you want to use.

# if not DISABLE_LANGUAGE_MODEL and not API_KEY:
#   raise ValueError('API_KEY is required.')

# model = language_model_utils.language_model_setup(
#     api_type=API_TYPE,
#     model_name=MODEL_NAME,
#     api_key=API_KEY,
#     disable_language_model=DISABLE_LANGUAGE_MODEL,
# )

# Updated to use the profiler.

from concordia.language_model import profiled_language_model
from concordia.utils import profiler

profiler.reset()
profiler.enable()

base_model = language_model_utils.language_model_setup(
    api_type=API_TYPE,
    model_name=MODEL_NAME,
    api_key=API_KEY or None,
    disable_language_model=DISABLE_LANGUAGE_MODEL,
)

model = profiled_language_model.ProfiledLanguageModel(
    base_model,
    model_name=MODEL_NAME,
)


In [6]:
# @title Setup sentence encoder

if DISABLE_LANGUAGE_MODEL:
  embedder = lambda _: np.ones(3)
else:
  st_model = sentence_transformers.SentenceTransformer(
      'sentence-transformers/all-mpnet-base-v2'
  )
  embedder = lambda x: st_model.encode(x, show_progress_bar=False)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6370.24it/s]


In [7]:
test = model.sample_text(
    'Is societal and technological progress like getting a clearer picture of '
    'something true and deep?'
)
print(test)

Sometimes—but not necessarily.

Progress can make parts of reality clearer: better science helps us explain nature, and better institutions can reveal whose needs or experiences were previously ignored. In that sense, it can feel like bringing a hidden picture into focus.

But there’s no guarantee that history is steadily uncovering one complete, timeless truth. New tools and ideas also change what we notice, what questions we ask, and what we count as important. They can illuminate one aspect while obscuring another, and technological capability can grow faster than wisdom.

A useful way to put it is: progress can improve our maps of the world, but maps are not the territory. Whether a clearer map leads to a better life depends on what we choose to do with it.


In [8]:
# @title Load prefabs from packages to make the specific palette to use here.

prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

In [9]:
# @title Print menu of prefabs

display.display(
    display.Markdown(helper_functions.print_pretty_prefabs(prefabs))
)

---
**`basic__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?"',
    params={'name': 'Alice', 'goal': '', 'extra_components': {}, 'extra_components_index': {}, 'randomize_choices': True, 'prefix_entity_name': True, 'observation_history_length': 1000000, 'situation_perception_history_length': 25, 'self_perception_history_length': 1000000, 'person_by_situation_history_length': 5}
)
```
---
**`basic_scripted__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?"',
    params={'name': 'Alice', 'goal': '', 'script': []}
)
```
---
**`basic_with_plan__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?" and building a plan based on the answers. It then tries to execute the plan.',
    params={'name': 'Alice', 'goal': '', 'force_time_horizon': False}
)
```
---
**`conversational__Entity`**:
```python
Entity(
    description='An entity that participates in conversations, aiming to create a dynamically balanced and engaging dialogue.',
    params={'name': 'Debra'}
)
```
---
**`fake_assistant_with_configurable_system_prompt__Entity`**:
```python
Entity(
    description='An entity that simulates an AI assistant with a configurable system prompt.',
    params={'name': 'Assistant', 'system_prompt': 'Assistant is a helpful and harmless AI assistant.'}
)
```
---
**`minimal__Entity`**:
```python
Entity(
    description='An entity that has a minimal set of components and is configurable by the user. The initial set of components manage memory, observations, and instructions. If goal is specified, the entity will have a goal constant component.',
    params={'name': 'Alice', 'goal': '', 'custom_instructions': '', 'extra_components': {}, 'extra_components_index': {}, 'randomize_choices': True}
)
```
---
**`puppet__Entity`**:
```python
Entity(
    description='An entity with fixed responses for specific calls to action.',
    params={'name': 'Puppet Agent', 'fixed_responses': {}, 'goal': ''}
)
```
---
**`rational__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "Which options are available to me?", and "Of those options, which is best?"',
    params={'name': 'Rational Agent', 'goal': '', 'randomize_choices': True, 'prefix_entity_name': True, 'observation_history_length': 1000000, 'situation_perception_history_length': 25, 'available_options_history_length': 1000000, 'best_option_history_length': 5}
)
```
---
**`async_social_media__GameMaster`**:
```python
GameMaster(
    description='A game master for asynchronous social media simulations.',
    params={'name': 'forum_rules', 'forum_name': 'Community Forum', 'call_to_action': 'What does {name} do on the forum? Respond in JSON format with one of:\n{{"action": "post", "author": "{name}", "title": "...", "content": "..."}}\n{{"action": "reply", "author": "{name}", "post_id": "...", "content": "..."}}\n{{"action": "upvote", "author": "{name}", "post_id": "..."}}\n{{"action": "downvote", "author": "{name}", "post_id": "..."}}\n', 'extra_components': {}, 'extra_components_index': {}}
)
```
---
**`async_social_media___NextActingEligiblePlayers`**:
```python
<concordia.prefabs.game_master.async_social_media._NextActingEligiblePlayers object at 0x129160050>
```
---
**`dialogic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling conversation.',
    params={'name': 'conversation rules', 'next_game_master_name': 'default rules', 'acting_order': 'game_master_choice', 'can_terminate_simulation': True}
)
```
---
**`dialogic_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling conversation. This game master is designed to be used with scenes.',
    params={'name': 'conversation rules', 'scenes': (), 'extra_components': {}, 'extra_components_index': {}, 'external_queue': None, 'allow_llm_fallback': True}
)
```
---
**`formative_memories_initializer__GameMaster`**:
```python
GameMaster(
    description='An initializer for all entities that generates formative memories from their childhood.',
    params={'name': 'initial setup rules', 'next_game_master_name': 'default rules', 'shared_memories': [], 'player_specific_context': {}, 'player_specific_memories': {}}
)
```
---
**`game_theoretic_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling matrix game. decisions, designed to be used with scenes.',
    params={'name': 'decision rules', 'extra_components': {}, 'extra_components_index': {}, 'scenes': (), 'action_to_scores': <function _default_action_to_scores at 0x11364ac40>, 'scores_to_observation': <function _default_scores_to_observation at 0x11364ada0>, 'external_queue': None}
)
```
---
**`generic__GameMaster`**:
```python
GameMaster(
    description='A general purpose game master.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'extra_components': {}, 'extra_components_index': {}, 'acting_order': 'game_master_choice', 'llm_chooses_next_game_master': False}
)
```
---
**`interrupt_driven__GameMaster`**:
```python
GameMaster(
    description='An interrupt-driven game master with selective attention.',
    params={'name': 'interrupt_driven_rules', 'start_time': '2026-01-01T09:00:00', 'event_tag_for_actions': 'action', 'max_ticks': None, 'call_to_action': 'What does {name} do next? Describe their action, then end your response with a JSON object specifying attention and schedule.\n\nJSON schema:\n{"tags": [...], "mask": [...], "timer": {"time": "...", "reason": "..."}}\n\nFields:\n  mask   - list of event-tag prefixes to monitor.\n           [""]  = pay attention to everything\n           []    = pay attention to nothing until timer fires\n           ["chat.", "alarm."]  = only matching prefixes\n  timer  - when to next check in.\n           time: duration like "0m", "5m", "30m", "1h", "1h30m"\n           until: absolute time like "8:00", "14:30" (24-hour clock)\n           reason: human-readable explanation\n           Use "time" for a delay from now, or "until" for a specific\n           clock time. If both are given, "until" takes priority.\n  tags   - (optional) extra event tags others can match on.\n\nExamples:\nI wave hello to everyone.\n{"mask": ["chat."], "timer": {"time": "30m", "reason": "check messages"}}\n\nI focus on my work in silence.\n{"mask": [], "timer": {"time": "2h", "reason": "deep focus"}, "tags": ["status.busy"]}\n\nI study until the library closes.\n{"mask": ["study."], "timer": {"until": "21:00", "reason": "library closes"}}\n', 'initial_events': [], 'initial_entity_states': {}, 'extra_components': {}, 'extra_components_index': {}}
)
```
---
**`interviewer__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'InterviewerGM', 'player_names': [], 'questionnaires': [], 'verbose': False}
)
```
---
**`marketplace__GameMaster`**:
```python
GameMaster(
    description='A generic Game Master that administers a marketplace.',
    params={'name': 'ExperimenterGM', 'experiment_component_class': None, 'experiment_component_init_kwargs': {}, 'llm_chooses_next_game_master': False}
)
```
---
**`open_ended_interviewer__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'InterviewerGM', 'player_names': [], 'questionnaires': [], 'sequence_of_events': [], 'embedder': None, 'verbose': False, 'llm_chooses_next_game_master': False}
)
```
---
**`physically_situated_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master for physical time/place simulations with scene support. Combines full world simulation with structured scene progressions.',
    params={'name': 'physical action rules', 'scenes': (), 'next_game_master_name': None, 'extra_event_resolution_steps': '', 'clock_description': "The passing of time can be conveyed using any convenient feature of the environment, e.g. a physical clock, the angle of the sun, extent of a candle's melting, phase of the moon, agricultural season, elapsed time since an event, etc. Whenever possible, try to track the day and year as well as the time within the day. To determine the passing of time, try to make reasonable inferences about the amount of time that would most likely have elapsed between the previous event and the latest event, taking into account the number of simulation steps taken.", 'start_time': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}, 'external_queue': None}
)
```
---
**`psychology_experiment__GameMaster`**:
```python
GameMaster(
    description='A generic Game Master that administers a psychology experiment defined by custom observation and action specification components.',
    params={'name': 'ExperimenterGM', 'scenes': (), 'experiment_component_class': None, 'experiment_component_init_kwargs': {}}
)
```
---
**`scripted__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'ScriptedGM', 'script': [], 'verbose': False}
)
```
---
**`situated__GameMaster`**:
```python
GameMaster(
    description='A general game master for games set in a specific location.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}, 'llm_chooses_next_game_master': False}
)
```
---
**`situated_in_time_and_place__GameMaster`**:
```python
GameMaster(
    description='A general game master for games situated in a physical time/place.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'clock_description': "The passing of time can be conveyed using any convenient feature of the environment, e.g. a physical clock, the angle of the sun, extent of a candle's melting, phase of the moon, agricultural season, elapsed time since an event, etc. Whenever possible, try to track the day and year as well as the time within the day. To determine the passing of time, try to make reasonable inferences about the amount of time that would most likely have elapsed between the previous event and the latest event, taking into account the number of simulation steps taken.", 'start_time': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}, 'llm_chooses_next_game_master': False}
)
```
---

In [10]:
# @title Configure instances.

instances = [
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Oliver Cromwell',
            'goal': 'become lord protector',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'King Charles I',
            'goal': 'avoid execution for treason',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'default rules',
            # Comma-separated list of thought chain steps.
            'extra_event_resolution_steps': '',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'default rules',
            'shared_memories': [
                'The king was captured by Parliamentary forces in 1646.',
                'Charles I was tried for treason and found guilty.',
            ],
        },
    ),
]

In [11]:
config = prefab_lib.Config(
    default_premise='Today is January 29, 1649.',
    default_max_steps=5,
    prefabs=prefabs,
    instances=instances,
)

# The simulation

In [12]:
# @title Initialize the simulation
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)

In [13]:
# @title Run the simulation
raw_log = []
# results_log = runnable_simulation.play(max_steps=5, raw_log=raw_log)
results_log = runnable_simulation.play(max_steps=2, raw_log=raw_log)

Terminate? No
Game master: initial setup rules
Entity Oliver Cromwell observed: The king was captured by Parliamentary forces in 1646.


Charles I was tried for treason and found guilty.


When Oliver Cromwell was 10 years old, he watched his father’s household fall quiet after a tenant arrived with news of a failed harvest. The man’s cap twisted in his hands as he spoke of rent he could not pay, and Oliver saw his father listen without anger but not without worry. That evening, the church bells sounded as they always had, yet Oliver heard them as a summons to more than prayer: someone had to answer for the people who suffered. He carried the tenant’s strained face with him long after the muddy lanes had dried.






When Oliver Cromwell was 17 years old, he left Huntingdon for Cambridge with books packed among the plain necessities of a student’s life. He expected lessons to sharpen his mind, but his father’s death soon called him home, and duty seemed to close around him before he ha

In [ ]:
# @title Display the log
# display.HTML(results_log.to_html())

stats = profiler.get_stats()

print("LLM calls:", stats["counters"].get("llm_calls_total", 0))
print("Estimated input tokens:",
      sum(stats["values"].get("llm_prompt_tokens", [])))
print("Estimated text-output tokens:",
      sum(stats["values"].get("llm_completion_tokens", [])))

LLM calls: 34
Estimated input tokens: 23087
Estimated text-output tokens: 4100


```
Copyright 2024 DeepMind Technologies Limited.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
```